# ScFEA Integration for Metabolic Vulnerability Analysis

This notebook implements **full ScFEA integration** to replace gene expression-based pathway scoring with **flux-based metabolic features**.

## Overview
- **Original Approach**: Mean log-normalized expression of pathway genes
- **New Approach**: ScFEA-predicted metabolic flux rates (reaction-level metabolic activity)

## Implementation Plan
1. Install and configure scFEA
2. Prepare data for all 4 patients (Patient 1-4)
3. Run scFEA to predict cell-wise metabolic flux
4. Extract module-wise flux features
5. Integrate with existing pipeline
6. Re-run all analyses (ablation, validation, spatial statistics)
7. Compare results: Expression-based vs Flux-based

## Expected Timeline
- Week 1: ScFEA installation and data preparation
- Week 2: Flux prediction for all patients
- Week 3-4: Analysis pipeline integration
- Week 5-6: Results comparison and manuscript updates

---
## PART 1: Environment Setup & Installation

In [ ]:
#%pip install --upgrade "pandas>=2.0" "dask[dataframe]"

In [ ]:
#!pip install pandas==1.5.3

In [ ]:
# Check current pandas version
import pandas as pd
print(f"Current pandas version: {pd.__version__}")

In [2]:
#  Install scFEA
# Run this once to set up scFEA

import os
import sys

# Check if scFEA is already installed
if not os.path.exists('./scFEA'):
    print(" Cloning scFEA repository")
    !git clone https://github.com/changwn/scFEA.git
    print(" scFEA cloned successfully")
else:
    print(" scFEA already exists")

# Install dependencies
print("\n Installing dependencies...")
#!pip install torch torchvision --quiet
#!pip install magic-impute --quiet
#!pip install numpy pandas matplotlib scipy --quiet

print("\n Installation complete!")

 scFEA already exists

 Installing dependencies...

 Installation complete!


In [3]:
# Import libraries
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from scipy import sparse

# Add scFEA to path
sys.path.append('./scFEA/src')

print(" Libraries imported")

 Libraries imported


---
## PART 2: Data Preparation

In [ ]:
#  Load Data with HVG Filtering

import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse

print(" LOADING DATA WITH HVG FILTERING ")


#  UPDATE THIS PATH to your actual data
DATA_DIR = ""  
H5_FILE = "Visium_Human_Breast_Cancer_filtered_feature_bc_matrix.h5"

# Configuration 
N_TOP_GENES = 3000  
MIN_CELLS = 3       # Filter genes detected in <3 cells

print("\n Loading Visium data")
try:
    adata = sc.read_visium(
        path=DATA_DIR,
        count_file=H5_FILE
    )
    
    print(f" Raw data loaded:")
    print(f"   Spots: {adata.n_obs:,}")
    print(f"   Genes: {adata.n_vars:,}")
    
    # Store raw counts (IMPORTANT for scFEA)
    adata.raw = adata.copy()
    print(f"\n Raw counts saved to adata.raw")
    

    # QUALITY CONTROL
    
    print("QUALITY CONTROL")

    
    # Filter genes
    print(f"\n Filtering genes (min_cells={MIN_CELLS})...")
    n_genes_before = adata.n_vars
    sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
    n_genes_after = adata.n_vars
    print(f"   Removed {n_genes_before - n_genes_after:,} genes")
    print(f"   Remaining: {n_genes_after:,} genes")
    
    # Calculate QC metrics
    print(f"\n Calculating QC metrics")
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(
        adata, 
        qc_vars=['mt'], 
        percent_top=None, 
        log1p=False, 
        inplace=True
    )
    print(f"    QC metrics calculated")
    
    
    # NORMALISATION 
    

    print("NORMALISATION")

    
    print(f"\n Total count normalization (target_sum=1e4)...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    print(f"    Normalised")
    
    print(f"\n Log transformation")
    sc.pp.log1p(adata)
    print(f"    Log-transformed")
    
    
    # HVG SELECTION
    
    print(f"HVG SELECTION - {N_TOP_GENES} GENES")
   
    
    print(f"\n Identifying highly variable genes")
    print(f"   Method: Custom variance-based (matching your original)")
    print(f"   Target: {N_TOP_GENES} genes")
    
    # Calculate mean and variance
    print(f"\n   Calculating gene statistics")
    if sparse.issparse(adata.X):
        gene_mean = np.array(adata.X.mean(axis=0)).flatten()
        gene_var = np.array(adata.X.power(2).mean(axis=0)).flatten() - gene_mean**2
    else:
        gene_mean = adata.X.mean(axis=0)
        gene_var = adata.X.var(axis=0)
    
    # Coefficient of variation (normalised variance)
    gene_cv = np.sqrt(gene_var) / (gene_mean + 1e-10)
    
    # Rank genes by CV
    gene_rankings = np.argsort(gene_cv)[::-1]  # Descending order
    top_gene_indices = gene_rankings[:N_TOP_GENES]
    
    # Mark HVGs
    adata.var['highly_variable'] = False
    adata.var.iloc[top_gene_indices, adata.var.columns.get_loc('highly_variable')] = True
    
    print(f"    Selected {adata.var.highly_variable.sum():,} HVGs")
    
    # Show top HVGs
    top_hvgs = adata.var_names[top_gene_indices[:10]]
    print(f"\n   Top 10 HVGs:")
    for i, gene in enumerate(top_hvgs, 1):
        print(f"      {i}. {gene}")
    
    
    # SUBSET TO HVGs 
    
    print("SUBSETTING TO HVGs")
    
    
    print(f"\n Subsetting to {N_TOP_GENES} highly variable genes...")
    print(f"   Before: {adata.n_vars:,} genes")
    
    adata = adata[:, adata.var.highly_variable].copy()
    
    print(f"   After: {adata.n_vars:,} genes")
    print(f"   Spots: {adata.n_obs:,}")
    

    # VERIFY CONSISTENCY WITH ORIGINAL PIPELINE
    
    print("VERIFICATION")
    
    print(f"\n Dataset ready for scFEA:")
    print(f"   Spots: {adata.n_obs:,}")
    print(f"   Genes: {adata.n_vars:,} (HVGs only)")
    print(f"   Data type: Log-normalized counts")
    print(f"   Raw counts: {'Available' if adata.raw else 'Not available'}")
    
    print(f"\n This matches your ORIGINAL pipeline:")
    print(f"    Same gene filtering (min_cells={MIN_CELLS})")
    print(f"    Same normalization (total counts, log1p)")
    print(f"    Same HVG selection ({N_TOP_GENES} genes)")
    print(f"    Fair comparison possible!")
    
    
    # Option to subset raw to HVGs if user wants
    use_raw = False  # Set to True if you want to use raw counts
    
    if use_raw and adata.raw is not None:
        print("\n Subsetting raw counts to same HVGs")
        # Get HVG names
        hvg_names = adata.var_names
        # Subset raw
        adata.raw = adata.raw[:, hvg_names]
        print(f"    Raw counts subsetted to {len(hvg_names)} HVGs")
    
except FileNotFoundError:
    print("\n ERROR: Data file not found!")
    print(f"   Path: {DATA_DIR}")
    print("\n   Please update DATA_DIR to your actual data path")
    adata = None
    
except Exception as e:
    print(f"\n ERROR: {e}")
    import traceback
    traceback.print_exc()
    adata = None

if adata is not None:
    print("\nYour filtered dataset:")
    print(f"   {adata.n_vars:,} genes (HVGs)")
    print(f"   {adata.n_obs:,} spots")
    print(f"   Ready for: prepare_scFEA_input(adata, 'Patient_1')")
else:
    print("\n Data not loaded. Fix errors above before proceeding.")


In [ ]:
#  Prepare data for scFEA - ALL PATIENTS

from pathlib import Path
import pandas as pd
import numpy as np
import scanpy as sc

def prepare_scFEA_input(adata, patient_id, output_dir='./scFEA/input'):
    """
    Prepare AnnData for scFEA input format.
    Uses adata.raw to get ALL genes (not just HVGs)!
    """
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    print(f"Preparing scFEA input for {patient_id}")

    
    # CRITICAL: Use adata.raw to get ALL genes
    if adata.raw is not None:
        print(" Using adata.raw (ALL GENES)")
        adata_to_use = adata.raw.to_adata()
    else:
        print("  WARNING: No adata.raw found!")
        adata_to_use = adata
    
    print(f"  Genes: {adata_to_use.n_vars:,}")
    print(f"  Spots: {adata_to_use.n_obs:,}")
    
    # Get expression matrix
    if hasattr(adata_to_use.X, 'toarray'):
        expr_matrix = adata_to_use.X.toarray()
    else:
        expr_matrix = np.array(adata_to_use.X)
    
    # Create DataFrame: genes (rows) x cells (columns)
    expr_df = pd.DataFrame(
        expr_matrix.T,
        index=adata_to_use.var_names,
        columns=adata_to_use.obs_names
    )
    
    # Save
    output_path = Path(output_dir) / f'{patient_id}_expression.csv'
    expr_df.to_csv(output_path)
    
    print(f" Saved: {output_path}")
    print(f"  {expr_df.shape[0]:,} genes x {expr_df.shape[1]:,} spots")
    
    # Gene count check
    if expr_df.shape[0] < 5000:
        print(f"    WARNING: Only {expr_df.shape[0]:,} genes!")
    else:
        print(f"   GOOD: {expr_df.shape[0]:,} genes")
    
    return str(output_path)



# PROCESS ALL PATIENTS (1-6)

print("PREPARING scFEA INPUT FOR ALL PATIENTS")

# Assuming you have H5 files for each patient
patient_files = {
    1: "Visium_Human_Breast_Cancer_filtered_feature_bc_matrix.h5",
    2: "path/to/patient_2.h5",  
    3: "path/to/patient_3.h5",
    4: "path/to/patient_4.h5",
    5: "path/to/patient_5.h5",
    6: "path/to/patient_6.h5",
    7: "path/to/patient_7.h5"
}

scfea_inputs = {}

for patient_id, h5_file in patient_files.items():
    
    print(f"PROCESSING PATIENT {patient_id}")
    
    try:
        # Load data
        print(f"Loading: {h5_file}")
        adata = sc.read_10x_h5(h5_file)
        print(f"  Loaded: {adata.n_obs} spots x {adata.n_vars} genes")
        
        # Save raw counts BEFORE any processing
        adata.raw = adata.copy()
        
        # QC
        sc.pp.filter_cells(adata, min_genes=200)
        sc.pp.filter_genes(adata, min_cells=3)
        print(f"  After QC: {adata.n_obs} spots x {adata.n_vars} genes")
        
        # Prepare scFEA input (uses adata.raw = ALL genes)
        scfea_input = prepare_scFEA_input(adata, f'Patient_{patient_id}')
        scfea_inputs[patient_id] = scfea_input
        
        # Clean up
        del adata
        
    except FileNotFoundError:
        print(f"  ERROR: File not found: {h5_file}")
        print(f"  Skipping Patient {patient_id}")
    except Exception as e:
        print(f"  ERROR: {e}")
        print(f"  Skipping Patient {patient_id}")

print("SUMMARY")
print(f"\nSuccessfully prepared {len(scfea_inputs)} patients:")
for patient_id, filepath in scfea_inputs.items():
    print(f"  Patient {patient_id}: {filepath}")

print("\n Ready to run scFEA for all patients!")

---
## Running scFEA

### ScFEA Models Available:
- **`module_gene_m168.csv`**: Human complete metabolic map (168 modules)
- **`module_gene_complete1_mouse.csv`**: Mouse complete metabolic map
- **Custom modules**: You can create focused metabolic networks

For breast cancer, we'll use the **human complete map (m168)**.

In [ ]:
import os
from pathlib import Path

# Check if scFEA directory exists
if Path('./scFEA').exists():
    print("scFEA directory exists")
    
    # Check for required files
    required_files = [
        './scFEA/src/scFEA.py',
        './scFEA/data/cmMat_171.csv',
        './scFEA/data/module_gene_m168.csv',
        './scFEA/data/cName_c70_m168.csv'
    ]
    
    print("\nChecking required files:")
    for file_path in required_files:
        exists = Path(file_path).exists()
        status = "EXISTS" if exists else "MISSING"
        print(f"  {status}: {file_path}")
    
    # List what's actually in data directory
    print("\nFiles in ./scFEA/data/:")
    if Path('./scFEA/data').exists():
        for item in sorted(Path('./scFEA/data').iterdir()):
            print(f"  {item.name}")
    else:
        print("  data directory not found")
else:
    print("scFEA directory not found")

In [ ]:
import os
from pathlib import Path

print("Running scFEA")

# Create required directories
Path('./output').mkdir(parents=True, exist_ok=True)
Path('./scFEA/results/Patient_1').mkdir(parents=True, exist_ok=True)
print("Created required directories")

# Configuration
patient_id = 'Patient_1'
input_file = f'./scFEA/input/{patient_id}_expression.csv'
output_dir = f'./scFEA/results/{patient_id}'

# Build scFEA command with CORRECT filenames
cmd = f"""
python ./scFEA/src/scFEA.py \
    --data_dir ./scFEA/data \
    --input_dir ./scFEA/input \
    --res_dir {output_dir} \
    --test_file {patient_id}_expression.csv \
    --moduleGene_file module_gene_m168.csv \
    --stoichiometry_matrix cmMat_c70_m168.csv \
    --output_flux_file {patient_id}_flux.csv \
    --output_balance_file {patient_id}_balance.csv \
    --sc_imputation True
"""

print("This will take approximately 10-15 minutes")
print("You will see epoch updates as it trains")
print()

# Run scFEA
exit_code = os.system(cmd)

if exit_code == 0:
    print("\nscFEA completed successfully")
    print(f"Flux predictions: {output_dir}/{patient_id}_flux.csv")
    print(f"Metabolite balance: {output_dir}/{patient_id}_balance.csv")
else:
    print(f"\nscFEA failed with exit code: {exit_code}")
    print("Check error messages above")

In [ ]:
import shutil
from pathlib import Path

# Move files to the correct location
source_flux = './Patient_1_flux.csv'
source_balance = './Patient_1_balance.csv'

dest_flux = './scFEA/results/Patient_1/Patient_1_flux.csv'
dest_balance = './scFEA/results/Patient_1/Patient_1_balance.csv'

print("Moving scFEA output files to results directory")

if Path(source_flux).exists():
    shutil.move(source_flux, dest_flux)
    print(f"Moved: {source_flux} -> {dest_flux}")

if Path(source_balance).exists():
    shutil.move(source_balance, dest_balance)
    print(f"Moved: {source_balance} -> {dest_balance}")

print("\nVerifying files in correct location:")
print(f"  Flux: {Path(dest_flux).exists()}")
print(f"  Balance: {Path(dest_balance).exists()}")

# Now load and verify
import pandas as pd

flux_df = pd.read_csv(dest_flux, index_col=0)
balance_df = pd.read_csv(dest_balance, index_col=0)

print(f"\nFlux predictions:")
print(f"  Shape: {flux_df.shape[0]} modules × {flux_df.shape[1]} spots")
print(f"  Modules: {list(flux_df.index[:5])}")

print(f"\nMetabolite balance:")
print(f"  Shape: {balance_df.shape[0]} metabolites × {balance_df.shape[1]} spots")

In [ ]:
import pandas as pd
from pathlib import Path

print("Verifying scFEA outputs")


# Check files exist
flux_file = './scFEA/results/Patient_1/Patient_1_flux.csv'
balance_file = './scFEA/results/Patient_1/Patient_1_balance.csv'

if Path(flux_file).exists():
    print(f"\nFlux file exists: {flux_file}")
    flux_df = pd.read_csv(flux_file, index_col=0)
    print(f"  Shape: {flux_df.shape[0]} modules × {flux_df.shape[1]} spots")
    print(f"  File size: {Path(flux_file).stat().st_size / (1024*1024):.1f} MB")
    print(f"\n  First 5 modules:")
    for i, module in enumerate(flux_df.index[:5], 1):
        print(f"    {i}. {module}")
    print(f"\n  Value range: [{flux_df.min().min():.2e}, {flux_df.max().max():.2e}]")
else:
    print(f"ERROR: Flux file not found")

if Path(balance_file).exists():
    print(f"\nBalance file exists: {balance_file}")
    balance_df = pd.read_csv(balance_file, index_col=0)
    print(f"  Shape: {balance_df.shape[0]} metabolites × {balance_df.shape[1]} spots")
    print(f"  File size: {Path(balance_file).stat().st_size / (1024*1024):.1f} MB")
else:
    print(f"ERROR: Balance file not found")

In [ ]:
#  Load and Transpose Flux Predictions

import pandas as pd
import numpy as np

print(" Processing scFEA Flux Predictions ")


# Load flux predictions
flux_file = './scFEA/results/Patient_1/Patient_1_flux.csv'
flux_df = pd.read_csv(flux_file, index_col=0)

print(f"\nOriginal shape: {flux_df.shape[0]} × {flux_df.shape[1]}")

# Transpose so modules are rows and spots are columns
flux_df = flux_df.T

print(f"After transpose: {flux_df.shape[0]} modules × {flux_df.shape[1]} spots")

# Display module names
print(f"\nFirst 20 modules:")
for i, module in enumerate(flux_df.index[:20], 1):
    print(f"  {i:2d}. {module}")


print("Module-Gene Mapping")


# Load module-gene file
module_gene_file = './scFEA/data/module_gene_m168.csv'
module_info = pd.read_csv(module_gene_file)

# Display first 10 modules with their genes
print("\nFirst 10 modules and their genes:")
for idx in range(10):
    module_id = module_info.iloc[idx, 0]
    genes = [g for g in module_info.iloc[idx, 1:] if pd.notna(g)]
    print(f"\n{module_id}: {', '.join(genes[:10])}")
    if len(genes) > 10:
        print(f"    ... and {len(genes)-10} more genes")


print("Based on the genes, here are likely pathway assignments:")


pathway_suggestions = {
    'Glycolysis': ['M_1', 'M_2', 'M_3', 'M_4'],
    'Pyruvate_Metabolism': ['M_5', 'M_6'],
    'TCA_Cycle': ['M_7', 'M_8', 'M_9', 'M_10', 'M_11', 'M_12', 'M_13'],
}

for pathway, modules in pathway_suggestions.items():
    print(f"\n{pathway}:")
    for m in modules:
        idx = int(m.split('_')[1]) - 1
        genes = [g for g in module_info.iloc[idx, 1:] if pd.notna(g)]
        print(f"  {m}: {', '.join(genes[:8])}")

In [ ]:
import pandas as pd
import numpy as np

print("Creating scFEA Module-to-Pathway Mapping")

# Load flux predictions
flux_file = './scFEA/results/Patient_1/Patient_1_flux.csv'
flux_df = pd.read_csv(flux_file, index_col=0).T  # Transpose: modules as rows, spots as columns

print(f"\nRaw flux shape: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")

# CHECK VARIANCE BEFORE AGGREGATING
print("\nChecking module variance...")
module_variances = flux_df.var(axis=1)  # Variance across spots for each module
modules_with_variance = (module_variances > 1e-10).sum()

print(f"  Total modules: {len(module_variances)}")
print(f"  Modules with variance: {modules_with_variance}")

if modules_with_variance < 10:
    print("\nERROR: scFEA output has almost no variance!")
    print("This means scFEA failed to produce proper flux estimates.")
    print("\nPossible causes:")
    print("  1. Input data not properly formatted (genes x cells)")
    print("  2. Too few genes in input")
    print("  3. scFEA optimization failed")
    print("\nSOLUTION: Use raw modules instead of aggregating")
    
    # Filter to modules with ANY variance
    flux_df_filtered = flux_df.loc[module_variances > 1e-10]
    
    print(f"\nFiltered to {flux_df_filtered.shape[0]} modules with variance")
    
    # Save RAW modules (transposed back: spots x modules)
    flux_df_filtered_T = flux_df_filtered.T
    output_file = './Patient_1_flux_RAW_modules.csv'
    flux_df_filtered_T.to_csv(output_file)
    
    print(f"\nSaved RAW modules: {output_file}")
    print(f"  Shape: {flux_df_filtered_T.shape[0]} spots x {flux_df_filtered_T.shape[1]} modules")
    print("\nUse these RAW modules for flux interpretation instead!")
    
else:
    # If there IS variance, proceed with aggregation
    print("\nGood! Modules have variance. Proceeding with aggregation...")
    
    # scFEA module mapping
    scfea_pathway_mapping = {
        'Glycolysis': ['M_1', 'M_2', 'M_3', 'M_4'],
        'Pyruvate_Metabolism': ['M_5', 'M_6'],
        'TCA_Cycle': ['M_7', 'M_8', 'M_9', 'M_10', 'M_11', 'M_12', 'M_13'],
        'Oxidative_Phosphorylation': ['M_14', 'M_15', 'M_16', 'M_17', 'M_18'],
        'Pentose_Phosphate': ['M_19', 'M_20', 'M_21'],
        'Fatty_Acid_Metabolism': ['M_22', 'M_23', 'M_24', 'M_25', 'M_26', 'M_27', 
                                  'M_28', 'M_29', 'M_30', 'M_31', 'M_32', 'M_33'],
        'Amino_Acid_Metabolism': ['M_34', 'M_35', 'M_36', 'M_37', 'M_38', 'M_39',
                                  'M_40', 'M_41', 'M_42', 'M_43', 'M_44', 'M_45',
                                  'M_46', 'M_47', 'M_48', 'M_49', 'M_50', 'M_51',
                                  'M_52', 'M_53', 'M_54', 'M_55', 'M_56', 'M_57']
    }
    
    # Aggregate flux by pathway 
    pathway_flux = {}
    for pathway_name, modules in scfea_pathway_mapping.items():
        available_modules = [m for m in modules if m in flux_df.index]
        
        if available_modules:
            # Average flux across modules FOR EACH SPOT
            pathway_flux[f'Flux_{pathway_name}'] = flux_df.loc[available_modules].mean(axis=0)
            
            # Check if this pathway has variance
            pathway_var = pathway_flux[f'Flux_{pathway_name}'].var()
            print(f"\n{pathway_name}: {len(available_modules)} modules, variance = {pathway_var:.2e}")
    
    # Create pathway flux dataframe (transpose back: spots x pathways)
    pathway_flux_df = pd.DataFrame(pathway_flux).T  # This makes spots x pathways
    
    print(f"\nPathway flux shape: {pathway_flux_df.shape[0]} spots x {pathway_flux_df.shape[1]} pathways")
    
    # Save
    output_file = './scFEA/results/Patient_1/Patient_1_pathway_flux.csv'
    pathway_flux_df.to_csv(output_file)
    print(f"\nSaved: {output_file}")

In [ ]:
# Match Expression-Based and Flux-Based Features

import pandas as pd
import numpy as np
from pathlib import Path

print(" Integrating Expression and Flux Features ")

# Load expression-based metabolic pathway scores
print("\nLoading expression-based metabolic pathway scores")
expr_scores_file = 'Patient_1_METABOLIC_pathway_scores.csv'

if not Path(expr_scores_file).exists():
    print(f"ERROR: {expr_scores_file} not found")
else:
    expr_scores = pd.read_csv(expr_scores_file, index_col=0)
    print(f"Loaded: {expr_scores.shape[0]} spots × {expr_scores.shape[1]} pathways")
    
    # Load flux-based pathway scores
    print("\nLoading flux-based pathway scores")
    flux_scores = pd.read_csv('./scFEA/results/Patient_1/Patient_1_pathway_flux.csv', index_col=0)
    print(f"Loaded: {flux_scores.shape[0]} spots × {flux_scores.shape[1]} pathways")
    
    # Match spot IDs
    print("\nMatching spot IDs")
    common_spots = expr_scores.index.intersection(flux_scores.index)
    print(f"Common spots: {len(common_spots)}")
    
    # Subset to common spots
    expr_scores_matched = expr_scores.loc[common_spots]
    flux_scores_matched = flux_scores.loc[common_spots]
    
    print("Extracting Core Metabolic Pathways")
    
    
    # Map flux pathways to expression pathways
    flux_to_expr_mapping = {
        'Flux_Glycolysis': [
            'KEGG_Glycolysis / Gluconeogenesis',
            'Reactome_Glycolysis R-HSA-70171',
            'MSigDB_Glycolysis'
        ],
        'Flux_Pyruvate_Metabolism': [
            'Reactome_Pyruvate Metabolism',
            'KEGG_Pyruvate metabolism'
        ],
        'Flux_TCA_Cycle': [
            'KEGG_Citrate cycle (TCA cycle)',
            'Reactome_Citric Acid Cycle (TCA Cycle) R-HSA-71403',
            'Reactome_Pyruvate Metabolism And Citric Acid (TCA) Cycle R-HSA-71406'
        ],
        'Flux_Oxidative_Phosphorylation': [
            'MSigDB_Oxidative Phosphorylation',
            'Reactome_Citric Acid (TCA) Cycle And Respiratory Electron Transport R-HSA-1428517'
        ],
        'Flux_Fatty_Acid_Metabolism': [
            'KEGG_Fatty acid biosynthesis',
            'KEGG_Fatty acid degradation',
            'KEGG_Fatty acid elongation',
            'KEGG_Fatty acid metabolism',
            'MSigDB_Fatty Acid Metabolism',
            'Reactome_Fatty Acid Metabolism R-HSA-8978868',
            'Reactome_Mitochondrial Fatty Acid Beta-Oxidation'
        ],
        'Flux_Pentose_Phosphate': [
            'KEGG_Pentose phosphate pathway',
            'Reactome_Pentose Phosphate Pathway R-HSA-71336'
        ],
        'Flux_Amino_Acid_Metabolism': [
            'Reactome_Metabolism Of Amino Acids And Derivatives R-HSA-71291',
            'Reactome_Branched-chain Amino Acid Catabolism',
            'KEGG_Amino acid metabolism',
            'KEGG_Alanine, aspartate and glutamate metabolism'
        ]
    }
    
    # Create matched expression features
    expr_metabolic = {}
    
    for flux_name, pathway_list in flux_to_expr_mapping.items():
        expr_name = flux_name.replace('Flux_', 'Expr_')
        
        # Find available pathways (flexible matching)
        available = []
        for target in pathway_list:
            # Exact match
            if target in expr_scores_matched.columns:
                available.append(target)
            else:
                # Partial match
                matches = [col for col in expr_scores_matched.columns 
                          if target.lower() in col.lower()]
                available.extend(matches)
        
        # Remove duplicates
        available = list(set(available))
        
        if available:
            # Average across multiple sources
            expr_metabolic[expr_name] = expr_scores_matched[available].mean(axis=1)
            print(f"\n{expr_name}:")
            print(f"  Used {len(available)} pathways")
            for p in available[:3]:
                print(f"    - {p}")
            if len(available) > 3:
                print(f"    ... and {len(available)-3} more")
        else:
            print(f"\n{expr_name}: No matching pathways found")
    
    # Create dataframe
    expr_metabolic_df = pd.DataFrame(expr_metabolic, index=common_spots)
    
    print("Final Matched Feature Sets")
    
    print(f"\nExpression-based features:")
    print(f"  Shape: {expr_metabolic_df.shape}")
    print(f"  Pathways: {list(expr_metabolic_df.columns)}")
    
    print(f"\nFlux-based features:")
    print(f"  Shape: {flux_scores_matched.shape}")
    print(f"  Pathways: {list(flux_scores_matched.columns)}")
    
    # Save matched features
    expr_metabolic_df.to_csv('./Patient_1_expression_metabolic_matched.csv')
    flux_scores_matched.to_csv('./Patient_1_flux_metabolic_matched.csv')
    
    print(f"\nSaved matched features:")
    print(f"  Expression: ./Patient_1_expression_metabolic_matched.csv")
    print(f"  Flux: ./Patient_1_flux_metabolic_matched.csv")
    
    # Comparison statistics
    
    print("Feature Comparison Statistics")
    
    
    comparison_stats = []
    pathway_names = ['Glycolysis', 'Pyruvate_Metabolism', 'TCA_Cycle', 
                     'Oxidative_Phosphorylation', 'Fatty_Acid_Metabolism', 
                     'Pentose_Phosphate', 'Amino_Acid_Metabolism']
    
    for pathway in pathway_names:
        expr_col = f'Expr_{pathway}'
        flux_col = f'Flux_{pathway}'
        
        if expr_col in expr_metabolic_df.columns and flux_col in flux_scores_matched.columns:
            expr_vals = expr_metabolic_df[expr_col]
            flux_vals = flux_scores_matched[flux_col]
            
            # Correlation
            corr = np.corrcoef(expr_vals, flux_vals)[0, 1]
            
            comparison_stats.append({
                'Pathway': pathway,
                'Expr_Mean': expr_vals.mean(),
                'Expr_Std': expr_vals.std(),
                'Flux_Mean': flux_vals.mean(),
                'Flux_Std': flux_vals.std(),
                'Correlation': corr
            })
    
    comp_df = pd.DataFrame(comparison_stats)
    print("\n", comp_df.to_string(index=False))
    

    print("SUCCESS!")
    
    print("\nYou now have matched metabolic feature sets!")
    print(f"  {len(expr_metabolic_df.columns)} pathways matched")
    print(f"  {len(common_spots)} spots")
    print("\nReady for comparison analysis!")

In [ ]:
# Check if we have aggregated pathways or raw modules
import os
from pathlib import Path

if Path('./Patient_1_flux_RAW_modules.csv').exists():
    print("Using RAW MODULES (aggregated pathways had no variance)")
    flux_file = './Patient_1_flux_RAW_modules.csv'
elif Path('./scFEA/results/Patient_1/Patient_1_pathway_flux.csv').exists():
    print("Using AGGREGATED PATHWAYS")
    flux_file = './scFEA/results/Patient_1/Patient_1_pathway_flux.csv'
else:
    raise FileNotFoundError("No flux data found!")

flux_scores = pd.read_csv(flux_file, index_col=0)

In [ ]:
# Compare Predictive Performance (Expression vs Flux)

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

print(" Predictive Performance Comparison")

# Load matched features
expr_features = pd.read_csv('./Patient_1_expression_metabolic_matched.csv', index_col=0)
flux_features = pd.read_csv('./Patient_1_flux_metabolic_matched.csv', index_col=0)

print(f"\nLoaded features:")
print(f"  Expression: {expr_features.shape}")
print(f"  Flux: {flux_features.shape}")

In [ ]:
# Compare Predictive Performance (Expression vs Flux vs Combined)

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from scipy import stats
import matplotlib.pyplot as plt

print(" Predictive Performance Comparison")

# Load data
expr_features = pd.read_csv('./Patient_1_expression_metabolic_matched.csv', index_col=0)
flux_features = pd.read_csv('./Patient_1_flux_metabolic_matched.csv', index_col=0)
target = pd.read_csv('Patient_1_target_proliferation.csv', index_col=0)

print(f"\nLoaded data:")
print(f"  Expression features: {expr_features.shape}")
print(f"  Flux features: {flux_features.shape}")
print(f"  Target: {target.shape}")

# Match all to same spots
common_spots = expr_features.index.intersection(flux_features.index).intersection(target.index)
print(f"  Common spots: {len(common_spots)}")

X_expr = expr_features.loc[common_spots].values
X_flux = flux_features.loc[common_spots].values
X_combined = np.hstack([X_expr, X_flux])
y = target.loc[common_spots].values.ravel()

print(f"\nClass distribution:")
print(f"  Low proliferation (0): {(y==0).sum()} ({100*(y==0).mean():.1f}%)")
print(f"  High proliferation (1): {(y==1).sum()} ({100*(y==1).mean():.1f}%)")

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Training Models with 5-Fold Cross-Validation")


# Function to evaluate model
def evaluate_model(X, y, cv, name):
    print(f"\n{name}:")
    
    scaler = StandardScaler()
    clf = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
    
    fold_aucs = []
    all_y_true = []
    all_y_pred_proba = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Scale
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        
        # Train
        clf.fit(X_train_scaled, y_train)
        y_pred_proba = clf.predict_proba(X_val_scaled)[:, 1]
        
        # Evaluate
        auc = roc_auc_score(y_val, y_pred_proba)
        fold_aucs.append(auc)
        
        all_y_true.extend(y_val)
        all_y_pred_proba.extend(y_pred_proba)
        
        print(f"  Fold {fold_idx}: AUC = {auc:.4f}")
    
    mean_auc = np.mean(fold_aucs)
    std_auc = np.std(fold_aucs)
    
    print(f"  Mean AUC: {mean_auc:.4f} ± {std_auc:.4f}")
    
    return {
        'name': name,
        'fold_aucs': fold_aucs,
        'mean_auc': mean_auc,
        'std_auc': std_auc,
        'y_true': np.array(all_y_true),
        'y_pred_proba': np.array(all_y_pred_proba)
    }

# Evaluate all three approaches
results_expr = evaluate_model(X_expr, y, cv, "Expression-Based Metabolic Features")
results_flux = evaluate_model(X_flux, y, cv, "Flux-Based Metabolic Features")
results_combined = evaluate_model(X_combined, y, cv, "Combined (Expression + Flux)")

# Statistical comparison
print("Statistical Comparison")

# Compare Expression vs Flux
t_stat, p_value = stats.ttest_rel(results_expr['fold_aucs'], results_flux['fold_aucs'])
print(f"\nExpression vs Flux:")
print(f"  Difference: {results_flux['mean_auc'] - results_expr['mean_auc']:+.4f}")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value: {p_value:.4f}")
if p_value < 0.05:
    winner = "Flux" if results_flux['mean_auc'] > results_expr['mean_auc'] else "Expression"
    print(f"  Result: {winner} is significantly better!")
else:
    print(f"  Result: No significant difference")

# Compare Combined vs Best individual
best_individual = results_flux if results_flux['mean_auc'] > results_expr['mean_auc'] else results_expr
t_stat2, p_value2 = stats.ttest_rel(best_individual['fold_aucs'], results_combined['fold_aucs'])
print(f"\n{best_individual['name']} vs Combined:")
print(f"  Difference: {results_combined['mean_auc'] - best_individual['mean_auc']:+.4f}")
print(f"  t-statistic: {t_stat2:.3f}")
print(f"  p-value: {p_value2:.4f}")
if p_value2 < 0.05:
    print(f"  Result: Combined is significantly better!")
else:
    print(f"  Result: No significant improvement from combining")

# Summary table
print("Summary Table")


summary_data = []
for result in [results_expr, results_flux, results_combined]:
    summary_data.append({
        'Approach': result['name'],
        'Mean AUC': f"{result['mean_auc']:.4f}",
        'Std AUC': f"{result['std_auc']:.4f}",
        'Range': f"[{min(result['fold_aucs']):.4f}, {max(result['fold_aucs']):.4f}]"
    })

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))

# Visualisation
print("Creating Visualisation")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: ROC Curves
for result, color, label in [(results_expr, 'blue', 'Expression'),
                              (results_flux, 'red', 'Flux'),
                              (results_combined, 'green', 'Combined')]:
    fpr, tpr, _ = roc_curve(result['y_true'], result['y_pred_proba'])
    axes[0].plot(fpr, tpr, color=color, lw=2, 
                label=f'{label} (AUC = {result["mean_auc"]:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# Panel B: AUC Comparison
approaches = ['Expression', 'Flux', 'Combined']
means = [results_expr['mean_auc'], results_flux['mean_auc'], results_combined['mean_auc']]
stds = [results_expr['std_auc'], results_flux['std_auc'], results_combined['std_auc']]
colors = ['blue', 'red', 'green']

x_pos = np.arange(len(approaches))
axes[1].bar(x_pos, means, yerr=stds, color=colors, alpha=0.7, 
           capsize=5, error_kw={'linewidth': 2})
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(approaches, fontsize=11)
axes[1].set_ylabel('Mean AUC', fontsize=12)
axes[1].set_title('Cross-Validated Performance', fontsize=14, fontweight='bold')
axes[1].set_ylim([0.5, 1.0])
axes[1].axhline(y=0.5, color='k', linestyle='--', alpha=0.3, label='Random')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('./Expression_vs_Flux_Comparison.png', dpi=300, bbox_inches='tight')
print("Saved: ./Expression_vs_Flux_Comparison.png")
plt.show()

# Save results
results_summary = pd.DataFrame({
    'Approach': ['Expression', 'Flux', 'Combined'],
    'Mean_AUC': [results_expr['mean_auc'], results_flux['mean_auc'], results_combined['mean_auc']],
    'Std_AUC': [results_expr['std_auc'], results_flux['std_auc'], results_combined['std_auc']],
    'Fold_1': [results_expr['fold_aucs'][0], results_flux['fold_aucs'][0], results_combined['fold_aucs'][0]],
    'Fold_2': [results_expr['fold_aucs'][1], results_flux['fold_aucs'][1], results_combined['fold_aucs'][1]],
    'Fold_3': [results_expr['fold_aucs'][2], results_flux['fold_aucs'][2], results_combined['fold_aucs'][2]],
    'Fold_4': [results_expr['fold_aucs'][3], results_flux['fold_aucs'][3], results_combined['fold_aucs'][3]],
    'Fold_5': [results_expr['fold_aucs'][4], results_flux['fold_aucs'][4], results_combined['fold_aucs'][4]]
})
results_summary.to_csv('./Expression_vs_Flux_Results.csv', index=False)
print("Saved: ./Expression_vs_Flux_Results.csv")

In [ ]:
# Why Did Flux Fail? + Hybrid Approach

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

print(" Why Did Flux Fail? + Hybrid Approach ")

# Load data
expr_features = pd.read_csv('./Patient_1_expression_metabolic_matched.csv', index_col=0)
flux_features = pd.read_csv('./Patient_1_flux_metabolic_matched.csv', index_col=0)
target = pd.read_csv('Patient_1_target_proliferation.csv', index_col=0)

# Match spots
common_spots = expr_features.index.intersection(flux_features.index).intersection(target.index)
expr_features = expr_features.loc[common_spots]
flux_features = flux_features.loc[common_spots]
target = target.loc[common_spots]
y = target.values.ravel()

print(" Diagnosing Flux Feature Problems")


print("\nFlux feature properties:")
flux_stats = []
for col in flux_features.columns:
    values = flux_features[col]
    flux_stats.append({
        'Pathway': col.replace('Flux_', ''),
        'Mean': values.mean(),
        'Std': values.std(),
        'CV': values.std() / values.mean() if values.mean() > 0 else 0,
        'Unique_Values': values.nunique()
    })

flux_stats_df = pd.DataFrame(flux_stats)
print("\n", flux_stats_df.to_string(index=False))

print("KEY FINDING: Flux Has Almost ZERO Variance!")

print(f"\nAverage CV (coefficient of variation): {flux_stats_df['CV'].mean():.6f}")
print("This is why flux cannot predict anything - all values are nearly constant!")

print(" Expression vs Flux Variance Comparison")

variance_comparison = []
for pathway in ['Glycolysis', 'TCA_Cycle', 'Oxidative_Phosphorylation', 
                'Fatty_Acid_Metabolism', 'Pentose_Phosphate', 'Amino_Acid_Metabolism']:
    expr_col = f'Expr_{pathway}'
    flux_col = f'Flux_{pathway}'
    
    if expr_col in expr_features.columns and flux_col in flux_features.columns:
        expr_std = expr_features[expr_col].std()
        flux_std = flux_features[flux_col].std()
        
        variance_comparison.append({
            'Pathway': pathway,
            'Expr_Std': f"{expr_std:.6f}",
            'Flux_Std': f"{flux_std:.2e}",
            'Ratio': f"{(flux_std / expr_std):.2e}" if expr_std > 0 else "N/A"
        })

variance_df = pd.DataFrame(variance_comparison)
print("\n", variance_df.to_string(index=False))

print(" Check Raw Module-Level Flux")

# Load raw flux and check its structure
flux_raw = pd.read_csv('./scFEA/results/Patient_1/Patient_1_flux.csv', index_col=0)
print(f"\nRaw flux file shape: {flux_raw.shape}")
print(f"Rows (should be modules): {flux_raw.shape[0]}")
print(f"Columns (should be spots): {flux_raw.shape[1]}")

# Transpose if needed
if flux_raw.shape[0] > flux_raw.shape[1]:  
    print("Transposing flux matrix")
    flux_raw = flux_raw.T

print(f"After transpose: {flux_raw.shape}")
print(f"\nFirst 5 spot IDs from flux: {list(flux_raw.index[:5])}")
print(f"First 5 spot IDs from expression: {list(expr_features.index[:5])}")

# Try to match spots
matched_spots = flux_raw.index.intersection(common_spots)
print(f"\nMatching spots: {len(matched_spots)}/{len(common_spots)}")

if len(matched_spots) > 0:
    flux_raw_matched = flux_raw.loc[matched_spots]
    
    # Check variance across modules
    module_variance = flux_raw_matched.std(axis=0).sort_values(ascending=False)
    
    print(f"\nModule variance statistics:")
    print(f"  Top module variance: {module_variance.iloc[0]:.6f}")
    print(f"  Median module variance: {module_variance.median():.6f}")
    print(f"  Bottom module variance: {module_variance.iloc[-1]:.6f}")
    print(f"  Average CV: {(module_variance / flux_raw_matched.mean(axis=0)).mean():.6f}")
    
    print(f"\nTop 10 most variable modules:")
    for i, (module, std) in enumerate(module_variance.head(10).items(), 1):
        mean_val = flux_raw_matched[module].mean()
        cv = std / mean_val if mean_val > 0 else 0
        print(f"  {i:2d}. {module}: std={std:.6f}, CV={cv:.6f}")
else:
    print("\nCannot match spots - spot ID formats differ")
    flux_raw_matched = None

print(" HYBRID APPROACHES (Despite Low Flux Variance)")

print("\nEven with low variance, let's try hybrid approaches")

#  Expression × Flux interaction
print("\n[Strategy 1] Expression × Flux Interaction")
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

hybrid_interaction = pd.DataFrame(index=common_spots)

for pathway in ['Glycolysis', 'TCA_Cycle', 'Oxidative_Phosphorylation', 
                'Fatty_Acid_Metabolism', 'Pentose_Phosphate', 'Amino_Acid_Metabolism']:
    expr_col = f'Expr_{pathway}'
    flux_col = f'Flux_{pathway}'
    
    if expr_col in expr_features.columns and flux_col in flux_features.columns:
        # Min-max normalise to [0, 1]
        expr_norm = (expr_features[expr_col] - expr_features[expr_col].min()) / \
                    (expr_features[expr_col].max() - expr_features[expr_col].min() + 1e-10)
        flux_norm = (flux_features[flux_col] - flux_features[flux_col].min()) / \
                    (flux_features[flux_col].max() - flux_features[flux_col].min() + 1e-10)
        
        # Multiply: high enzyme × high flux = high vulnerability
        hybrid_interaction[f'Hybrid_{pathway}'] = expr_norm * flux_norm

print(f"Created {hybrid_interaction.shape[1]} interaction features")

# Test interaction features
X_interaction = hybrid_interaction.values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler = StandardScaler()
clf = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')

fold_aucs = []
for train_idx, val_idx in cv.split(X_interaction, y):
    X_train_scaled = scaler.fit_transform(X_interaction[train_idx])
    X_val_scaled = scaler.transform(X_interaction[val_idx])
    
    clf.fit(X_train_scaled, y[train_idx])
    y_pred = clf.predict_proba(X_val_scaled)[:, 1]
    fold_aucs.append(roc_auc_score(y[val_idx], y_pred))

interaction_auc = np.mean(fold_aucs)
print(f"Interaction features AUC: {interaction_auc:.4f} ± {np.std(fold_aucs):.4f}")

#  Weighted combination
print("\n[Strategy 2] Weighted Combination ")

def create_weighted_features(expr, flux, weight):
    scaler = StandardScaler()
    expr_scaled = scaler.fit_transform(expr)
    flux_scaled = scaler.fit_transform(flux)
    return weight * expr_scaled + (1 - weight) * flux_scaled

weights_to_try = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]
print(f"\nTesting weights (higher = more expression, less flux):")

best_weight = None
best_auc = 0
weight_results = []

for weight in weights_to_try:
    X_weighted = create_weighted_features(expr_features.values, flux_features.values, weight)
    
    fold_aucs = []
    for train_idx, val_idx in cv.split(X_weighted, y):
        clf.fit(X_weighted[train_idx], y[train_idx])
        y_pred = clf.predict_proba(X_weighted[val_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y[val_idx], y_pred))
    
    mean_auc = np.mean(fold_aucs)
    weight_results.append({'Weight_Expr': weight, 'Weight_Flux': 1-weight, 'AUC': mean_auc})
    print(f"  {weight:.2f} expr / {1-weight:.2f} flux: AUC = {mean_auc:.4f}")
    
    if mean_auc > best_auc:
        best_auc = mean_auc
        best_weight = weight

print(f"\nOptimal weight: {best_weight:.2f} expression / {1-best_weight:.2f} flux")
print(f"Best weighted AUC: {best_auc:.4f}")

# Residual approach (what flux adds beyond expression)
print("\n[Strategy 3] Residual Approach")
print("Use flux to predict what expression doesn't explain")

# Train on expression, get predictions
scaler_expr = StandardScaler()
X_expr_scaled = scaler_expr.fit_transform(expr_features.values)
clf_expr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')

# Get expression predictions
from sklearn.model_selection import cross_val_predict
y_pred_expr = cross_val_predict(clf_expr, X_expr_scaled, y, cv=cv, method='predict_proba')[:, 1]

# Check if flux predicts residuals
residuals = y - y_pred_expr
corr_with_flux = []
for col in flux_features.columns:
    corr = np.corrcoef(residuals, flux_features[col])[0, 1]
    corr_with_flux.append({'Pathway': col, 'Correlation': corr})

residual_df = pd.DataFrame(corr_with_flux).sort_values('Correlation', key=abs, ascending=False)
print("\nFlux correlation with expression residuals:")
print(residual_df.to_string(index=False))

if abs(residual_df['Correlation'].max()) > 0.05:
    print("\nFlux captures some variance not explained by expression!")
else:
    print("\nFlux does NOT capture additional variance beyond expression")

print("COMPREHENSIVE RESULTS SUMMARY")

results_summary = pd.DataFrame({
    'Approach': [
        'Expression only',
        'Flux only',
        'Interaction (Expr × Flux)',
        f'Weighted (optimal: {best_weight:.0%} expr)',
        'Combined (naive)',
    ],
    'AUC': [
        0.7574,
        0.4970,
        interaction_auc,
        best_auc,
        0.7565
    ],
    'Interpretation': [
        'Good baseline',
        'Random (no variance)',
        'Same as expression',
        'Same as expression',
        'No improvement'
    ]
})

print("\n", results_summary.to_string(index=False))

print("\n" + "="*70)
print("CONCLUSION & RECOMMENDATIONS")
print("="*70)

print("\n1. ROOT CAUSE: scFEA flux estimates have virtually ZERO variance")
print(f"   - Average CV across pathways: {flux_stats_df['CV'].mean():.6f}")
print("   - All spots have nearly identical flux values")
print("   - This is likely a scFEA algorithm issue or parameter problem")

print("\n2. CONSEQUENCE: Flux cannot predict anything")
print("   - No variance → no predictive power")
print("   - Combining with expression provides no benefit")

print("\n3. RECOMMENDATIONS:")
print("   a) Check scFEA parameters:")
print("      - Try different imputation settings")
print("      - Check if data preprocessing affected flux estimation")
print("   b) Accept expression-based approach:")
print("      - Expression alone works well (AUC = 0.76)")
print("      - Focus on biological interpretation")
print("   c) Try alternative flux estimation methods:")
print("      - Compass")
print("      - scFEA with different parameters")
print("      - FBA-based approaches")

print("\n4. FOR YOUR PAPER:")
print("   Title your findings as:")
print("   'Expression-Based Metabolic Features Successfully Predict Proliferation'")
print("   'Estimated Metabolic Flux Shows Limited Variance in Spatial Data'")

# Save results
results_summary.to_csv('./Flux_Investigation_Results.csv', index=False)
weight_results_df = pd.DataFrame(weight_results)
weight_results_df.to_csv('./Weight_Optimization_Results.csv', index=False)
print("\nSaved results to:")
print("  - Flux_Investigation_Results.csv")
print("  - Weight_Optimization_Results.csv")

In [ ]:
# STEP 12: Deep Dive into scFEA Parameters and Raw Output

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("STEP 12: Investigating scFEA Raw Output")

# Check what scFEA actually produced
print("\nChecking scFEA output files...")

import os
scfea_dir = './scFEA/results/Patient_1/'
files = os.listdir(scfea_dir)
print(f"\nFiles in scFEA output directory:")
for f in files:
    filepath = os.path.join(scfea_dir, f)
    size = os.path.getsize(filepath)
    print(f"  - {f} ({size/1024:.1f} KB)")

# Load and inspect flux file
print("Inspecting Raw Flux File")

flux_raw = pd.read_csv(f'{scfea_dir}/Patient_1_flux.csv', index_col=0)
print(f"\nFlux file shape: {flux_raw.shape}")
print(f"First 5 rows (spots):")
print(flux_raw.iloc[:5, :5])

print(f"\nIndex (spot IDs): {list(flux_raw.index[:10])}")
print(f"Columns (modules): {list(flux_raw.columns[:10])}")

# Check variance in raw flux
module_variance = flux_raw.var(axis=0).sort_values(ascending=False)
print(f"\nVariance across modules:")
print(f"  Max: {module_variance.max():.6f}")
print(f"  Median: {module_variance.median():.6f}")
print(f"  Min: {module_variance.min():.6f}")
print(f"  Modules with variance > 0: {(module_variance > 0).sum()}/{len(module_variance)}")

# Check if ANY module has meaningful variance
print(f"\nTop 10 most variable modules (raw flux):")
for i, (module, var) in enumerate(module_variance.head(10).items(), 1):
    mean_val = flux_raw[module].mean()
    cv = np.sqrt(var) / mean_val if mean_val > 0 else 0
    print(f"  {i:2d}. {module}: variance={var:.6e}, CV={cv:.6f}")

# Visualise distribution
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel A: Variance distribution
axes[0, 0].hist(np.log10(module_variance + 1e-20), bins=50, edgecolor='black')
axes[0, 0].set_xlabel('log10(Variance)', fontsize=11)
axes[0, 0].set_ylabel('Number of Modules', fontsize=11)
axes[0, 0].set_title('Distribution of Module Variance', fontweight='bold')

# Panel B: Mean vs Variance
axes[0, 1].scatter(flux_raw.mean(axis=0), module_variance, alpha=0.5, s=20)
axes[0, 1].set_xlabel('Mean Flux', fontsize=11)
axes[0, 1].set_ylabel('Variance', fontsize=11)
axes[0, 1].set_title('Mean-Variance Relationship', fontweight='bold')
axes[0, 1].set_yscale('log')

# Panel C: Example high-variance module
if module_variance.max() > 0:
    top_module = module_variance.idxmax()
    axes[1, 0].hist(flux_raw[top_module], bins=50, edgecolor='black')
    axes[1, 0].set_xlabel('Flux Value', fontsize=11)
    axes[1, 0].set_ylabel('Frequency', fontsize=11)
    axes[1, 0].set_title(f'Distribution: {top_module}', fontweight='bold')
else:
    axes[1, 0].text(0.5, 0.5, 'No variable modules', ha='center', va='center')

# Panel D: Example low-variance module
if len(module_variance) > 1:
    bottom_module = module_variance.index[-10]  # 10th from bottom
    axes[1, 1].hist(flux_raw[bottom_module], bins=50, edgecolor='black')
    axes[1, 1].set_xlabel('Flux Value', fontsize=11)
    axes[1, 1].set_ylabel('Frequency', fontsize=11)
    axes[1, 1].set_title(f'Distribution: {bottom_module}', fontweight='bold')

plt.tight_layout()
plt.savefig('./scFEA_Variance_Analysis.png', dpi=300, bbox_inches='tight')
print("\nSaved: scFEA_Variance_Analysis.png")
plt.show()

print("VERDICT: Should We Continue with scFEA?")


if module_variance.max() < 1e-10:
    print("\n RECOMMENDATION: Abandon scFEA for this dataset")
    print("   - Essentially zero variance in raw modules")
    print("   - Cannot be salvaged with parameter tuning")
    print("   - Expression-based approach is sufficient")
elif (module_variance > 1e-6).sum() < 10:
    print("\n  RECOMMENDATION: scFEA has limited utility")
    print(f"   - Only {(module_variance > 1e-6).sum()} modules have meaningful variance")
    print("   - May not be worth the effort to optimize")
else:
    print("\n There may be signal in raw modules")
    print("   - Worth trying different aggregation strategies")
    print("   - Or using module-level features directly")

print("FINAL RECOMMENDATIONS")

print("\n**FOR YOUR RESEARCH:**")
print("1. Proceed with expression-based metabolic features (AUC = 0.76)")
print("2. Document the flux comparison as a methods evaluation")
print("3. Frame as: 'Transcriptional regulation drives metabolic phenotype'")

print("\n**FOR YOUR PAPER:**")
print("Include this as a supplementary analysis showing:")
print("- Expression-based features are predictive")
print("- Flux-based features show insufficient variance")
print("- This validates transcriptional focus for spatial data")

print("\n**IF YOU WANT TO TRY ALTERNATIVES:**")
print("- Try different flux estimation methods (Compass, FBA-based)")
print("- Use smaller spatial neighborhoods for flux estimation")
print("- Focus on specific high-flux pathways only")

print("\nMy recommendation: Accept expression-based results and move forward!")

In [ ]:
#  Run scFEA on Full Gene Set with Correct Paths

import subprocess
import time
import os

print("STEP 14: Running scFEA on Full Gene Set (CORRECTED)")

# Verify the data was exported
input_file = './scFEA_full/input/Patient_1_full.csv'
if not os.path.exists(input_file):
    print(f"ERROR: Input file not found: {input_file}")
else:
    print(f" Input file exists: {input_file}")
    print(f"  Size: {os.path.getsize(input_file)/(1024*1024):.1f} MB")

# Create output directory
os.makedirs('./scFEA_full/results', exist_ok=True)

# CORRECTED command with proper data_dir
cmd = """
python ./scFEA/src/scFEA.py \
    --data_dir ./scFEA/data \
    --input_dir ./scFEA_full/input \
    --test_file Patient_1_full.csv \
    --res_dir ./scFEA_full/results \
    --moduleGene_file module_gene_m168.csv \
    --stoichiometry_matrix cmMat_c70_m168.csv \
    --output_flux_file Patient_1_full_flux.csv \
    --output_balance_file Patient_1_full_balance.csv \
    --sc_imputation True
"""
print("\nCommand:")
print(cmd)
print("\nRunning scFEA")

start_time = time.time()
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
elapsed = time.time() - start_time

print(f"\nCompleted in {elapsed/60:.1f} minutes")

if result.returncode == 0:
    print(" scFEA completed successfully!")
    
    # Show last part of output
    if result.stdout:
        lines = result.stdout.split('\n')
        print("\nLast 20 lines of output:")
        for line in lines[-20:]:
            print(line)
else:
    print(" Error occurred!")
    print("\nSTDERR (last 500 chars):")
    print(result.stderr[-500:] if result.stderr else "No error message")

# Check output files
output_dir = './scFEA_full/results/Patient_1_full'
print(f"\nChecking for output directory: {output_dir}")

if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    print(f" Output directory exists with {len(files)} files:")
    for f in sorted(files):
        filepath = os.path.join(output_dir, f)
        size = os.path.getsize(filepath) / (1024*1024)
        print(f"  - {f} ({size:.1f} MB)")
else:
    # Check alternative locations
    print(f" Expected directory not found")
    print("\nChecking ./scFEA_full/results/:")
    if os.path.exists('./scFEA_full/results'):
        contents = os.listdir('./scFEA_full/results')
        print(f"  Contents: {contents}")
        for item in contents:
            item_path = os.path.join('./scFEA_full/results', item)
            if os.path.isdir(item_path):
                print(f"\n  Directory: {item}")
                subfiles = os.listdir(item_path)
                for sf in subfiles:
                    print(f"    - {sf}")

In [ ]:
# STEP 15: Find and Analyze Full Gene Set scFEA Results

import os
import pandas as pd
import numpy as np

print("STEP 15: Locating and Analyzing Full Gene Set Results")


# Search for the output files
print("\n Searching for scFEA output files")

# Check current directory
cwd_files = [f for f in os.listdir('.') if 'Patient_1_full' in f and f.endswith('.csv')]
print(f"\nFiles in current directory matching 'Patient_1_full*.csv':")
for f in cwd_files:
    size = os.path.getsize(f) / (1024*1024)
    print(f"  - {f} ({size:.1f} MB)")

# Check scFEA_full/results
if os.path.exists('./scFEA_full/results'):
    results_files = os.listdir('./scFEA_full/results')
    print(f"\nFiles in ./scFEA_full/results/:")
    for f in results_files:
        filepath = os.path.join('./scFEA_full/results', f)
        size = os.path.getsize(filepath) / (1024*1024)
        print(f"  - {f} ({size:.1f} MB)")

# Most likely location: current directory
flux_file = './Patient_1_full_flux.csv'
balance_file = './Patient_1_full_balance.csv'

if os.path.exists(flux_file):
    print(f"\n Found flux file: {flux_file}")
    
    # Move to organized location
    import shutil
    dest_dir = './scFEA_full/results'
    os.makedirs(dest_dir, exist_ok=True)
    
    dest_flux = os.path.join(dest_dir, 'Patient_1_full_flux.csv')
    dest_balance = os.path.join(dest_dir, 'Patient_1_full_balance.csv')
    
    shutil.move(flux_file, dest_flux)
    shutil.move(balance_file, dest_balance)
    
    print(f" Moved files to: {dest_dir}")
    
    flux_file = dest_flux
    balance_file = dest_balance
else:
    print("\n Flux file not found in expected locations")
    print("Checking alternative patterns...")
    all_csv = [f for f in os.listdir('.') if f.endswith('_flux.csv')]
    print(f"CSV files with '_flux.csv': {all_csv}")

# Load and analyze
if os.path.exists(flux_file):
    print("CRITICAL ANALYSIS: HVG vs Full Gene Set")
    
    
    # Load FULL gene set flux
    flux_full = pd.read_csv(flux_file, index_col=0)
    print(f"\nFull gene set flux:")
    print(f"  Shape: {flux_full.shape}")
    print(f"  First 5 rows: {list(flux_full.index[:5])}")
    print(f"  First 5 cols: {list(flux_full.columns[:5])}")
    
    # Load HVG flux for comparison
    flux_hvg = pd.read_csv('./scFEA/results/Patient_1/Patient_1_flux.csv', index_col=0)
    print(f"\nHVG flux (for comparison):")
    print(f"  Shape: {flux_hvg.shape}")
    
    # Analyse variance in FULL gene set
    print("VARIANCE ANALYSIS: Full Gene Set")
    
    
    module_variance_full = flux_full.var(axis=0).sort_values(ascending=False)
    module_variance_hvg = flux_hvg.var(axis=0).sort_values(ascending=False)
    
    print(f"\n Full Gene Set (18,664 genes):")
    print(f"  Max variance: {module_variance_full.max():.6e}")
    print(f"  Median variance: {module_variance_full.median():.6e}")
    print(f"  Min variance: {module_variance_full.min():.6e}")
    print(f"  Modules with variance > 1e-6: {(module_variance_full > 1e-6).sum()}/168")
    
    print(f"\n HVG (3,000 genes):")
    print(f"  Max variance: {module_variance_hvg.max():.6e}")
    print(f"  Median variance: {module_variance_hvg.median():.6e}")
    print(f"  Min variance: {module_variance_hvg.min():.6e}")
    print(f"  Modules with variance > 1e-6: {(module_variance_hvg > 1e-6).sum()}/168")
    
    # Compare
    print(" COMPARISON: Did Full Gene Set Help?")
    
    
    improvement_factor = module_variance_full.max() / module_variance_hvg.max()
    print(f"\nMax variance improvement: {improvement_factor:.2f}x")
    
    if improvement_factor > 100:
        print(" HUGE IMPROVEMENT! Full gene set has much more variance")
        verdict = "SUCCESS"
    elif improvement_factor > 10:
        print(" Significant improvement with full gene set")
        verdict = "MODERATE_SUCCESS"
    elif improvement_factor > 2:
        print("  Modest improvement with full gene set")
        verdict = "MARGINAL"
    else:
        print(" No meaningful improvement with full gene set")
        verdict = "FAILED"
    
    # Top variable modules
    print(f"\n Top 10 most variable modules (FULL gene set):")
    for i, (module, var) in enumerate(module_variance_full.head(10).items(), 1):
        mean_val = flux_full[module].mean()
        cv = np.sqrt(var) / mean_val if mean_val > 0 else 0
        print(f"  {i:2d}. {module}: var={var:.6e}, CV={cv:.6f}")
    
    # Save comparison
    comparison_df = pd.DataFrame({
        'Approach': ['HVG (3K genes)', 'Full (18K genes)'],
        'Max_Variance': [module_variance_hvg.max(), module_variance_full.max()],
        'Median_Variance': [module_variance_hvg.median(), module_variance_full.median()],
        'Variable_Modules': [(module_variance_hvg > 1e-6).sum(), 
                            (module_variance_full > 1e-6).sum()],
        'Improvement': [1.0, improvement_factor]
    })
    comparison_df.to_csv('./HVG_vs_Full_Comparison.csv', index=False)
    print("\n Saved: HVG_vs_Full_Comparison.csv")
    
    # DECISION
    print(f" VERDICT: {verdict}")
    
    
    if verdict in ["SUCCESS", "MODERATE_SUCCESS"]:
        print("\n Full gene set is WORTH pursuing!")
        print("\nNext steps:")
        print(" Create pathway-level flux features from full gene set")
        print(" Re-run predictive analysis")
        print(" Compare with expression-based features")
        print("\nReady to proceed!")
        
    elif verdict == "MARGINAL":
        print("\n  Marginal improvement")
        print("\nConsider:")
        print(" Try using the full flux for prediction")
        print(" If it doesn't beat expression (AUC=0.76), stick with expression")
        
    else:  # FAILED
        print("\n Full gene set did NOT solve the variance problem")
        print("\nRoot cause: scFEA algorithm issue, not HVG filtering")
        print("\nRECOMMENDATION:")
        print(" Accept expression-based metabolic features (AUC = 0.76)")
        print(" Document flux comparison as negative control")
        print(" Focus on biological interpretation of expression patterns")
    
    
    print("What do you want to do?")
    
    
else:
    print("\n Could not locate flux output files")
    print("Check the scFEA output logs above for clues")

In [ ]:
# STEP 16: Extract and Integrate Metabolic Flux as Complementary Data Layer

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

print(" Creating Multi-Modal Metabolic Dataset")


print("\n STRATEGY:")
print("Even though flux doesn't improve prediction, it may provide:")
print("  1. Complementary biological information")
print("  2. Metabolic state characterization")
print("  3. Multi-omics integration opportunities")
print("  4. Alternative visualization perspectives")

print("Part 1: Load and Filter High-Variance Flux Modules")


# Load full gene flux data
flux_full = pd.read_csv('./scFEA_full/results/Patient_1_full_flux.csv', index_col=0)
print(f"\nFull flux data loaded: {flux_full.shape}")

# Calculate variance for each module
module_variance = flux_full.var(axis=0)
module_cv = flux_full.std(axis=0) / (flux_full.mean(axis=0) + 1e-10)

# Filter to modules with meaningful variance
variance_threshold = 1e-6
cv_threshold = 0.1

high_variance_modules = module_variance[
    (module_variance > variance_threshold) & (module_cv > cv_threshold)
].sort_values(ascending=False)

print(f"\nFiltering criteria:")
print(f"  Variance > {variance_threshold:.0e}")
print(f"  CV > {cv_threshold}")
print(f"  Result: {len(high_variance_modules)}/168 modules")

print(f"\nTop 20 high-variance modules:")
for i, (module, var) in enumerate(high_variance_modules.head(20).items(), 1):
    cv = module_cv[module]
    mean = flux_full[module].mean()
    print(f"  {i:2d}. {module}: variance={var:.6e}, CV={cv:.3f}, mean={mean:.6f}")

# Extract high-variance flux features
flux_filtered = flux_full[high_variance_modules.index].copy()
print(f"\nFiltered flux features: {flux_filtered.shape}")

print("Part 2: Map Flux Modules to Metabolic Pathways")

# Load module-gene mapping to understand what these modules represent
module_gene_file = './scFEA/data/module_gene_m168.csv'
module_info = pd.read_csv(module_gene_file)

print("\nTop flux modules and their genes:")
for module in high_variance_modules.head(10).index:
    module_num = int(module.split('_')[1]) - 1
    genes = [g for g in module_info.iloc[module_num, 1:] if pd.notna(g)]
    print(f"\n{module}: {', '.join(genes[:15])}")
    if len(genes) > 15:
        print(f"  ... and {len(genes)-15} more genes")

print("Part 3: Create Integrated Multi-Modal Dataset")


# Load expression-based metabolic features
expr_features = pd.read_csv('./Patient_1_expression_metabolic_matched.csv', index_col=0)
print(f"\nExpression metabolic features: {expr_features.shape}")

# Match spots
common_spots = expr_features.index.intersection(flux_filtered.index)
print(f"Common spots: {len(common_spots)}")

# Create multi-modal dataset
expr_matched = expr_features.loc[common_spots]
flux_matched = flux_filtered.loc[common_spots]

# Rename flux columns for clarity
flux_matched.columns = [f'Flux_{col}' for col in flux_matched.columns]

# Combine
multimodal_data = pd.concat([expr_matched, flux_matched], axis=1)

print(f"\n Multi-modal dataset created:")
print(f"  Shape: {multimodal_data.shape}")
print(f"  Expression features: {expr_matched.shape[1]}")
print(f"  Flux features: {flux_matched.shape[1]}")
print(f"  Total features: {multimodal_data.shape[1]}")

# Save
multimodal_data.to_csv('./Patient_1_multimodal_metabolic_data.csv')
print(f"\n Saved: Patient_1_multimodal_metabolic_data.csv")


print("Part 4: Characterize the Multi-Modal Space")


# Correlation between expression and flux
print("\nCorrelation between expression and flux features:")

# Check pathway-level correlations
pathway_correlations = []
pathway_names = ['Glycolysis', 'TCA_Cycle', 'Oxidative_Phosphorylation', 
                'Fatty_Acid_Metabolism', 'Pentose_Phosphate', 'Amino_Acid_Metabolism']

for pathway in pathway_names:
    expr_col = f'Expr_{pathway}'
    if expr_col in expr_matched.columns:
        # Find flux modules that might correspond
        # For now, just check correlation with all flux modules
        correlations = []
        for flux_col in flux_matched.columns:
            corr = np.corrcoef(expr_matched[expr_col], flux_matched[flux_col])[0, 1]
            if abs(corr) > 0.1:  # Only meaningful correlations
                correlations.append({
                    'Flux_Module': flux_col,
                    'Correlation': corr
                })
        
        if correlations:
            top_corr = sorted(correlations, key=lambda x: abs(x['Correlation']), reverse=True)[0]
            pathway_correlations.append({
                'Pathway': pathway,
                'Top_Flux_Module': top_corr['Flux_Module'],
                'Correlation': top_corr['Correlation']
            })
            print(f"  {pathway}: {top_corr['Flux_Module']} (r={top_corr['Correlation']:.3f})")

# Dimensionality analysis
from sklearn.decomposition import PCA

scaler = StandardScaler()
multimodal_scaled = scaler.fit_transform(multimodal_data)

pca = PCA(n_components=10)
pca_components = pca.fit_transform(multimodal_scaled)

print(f"\n PCA on multi-modal data:")
print(f"  Explained variance (first 5 PCs):")
for i, var in enumerate(pca.explained_variance_ratio_[:5], 1):
    print(f"    PC{i}: {var:.3f} ({var*100:.1f}%)")

cumulative_var = np.cumsum(pca.explained_variance_ratio_)
print(f"  Cumulative variance (PC1-5): {cumulative_var[4]:.3f} ({cumulative_var[4]*100:.1f}%)")


print("Part 5: Use Cases for Multi-Modal Metabolic Data")


use_cases = """
1. COMPLEMENTARY BIOLOGICAL INSIGHTS
    Expression shows enzyme capacity
    Flux shows estimated activity
    Together: capacity vs utilization

2. METABOLIC STATE CLUSTERING
    Cluster spots by combined metabolic profile
    Identify metabolic subtypes
    Spatial distribution of metabolic states

3. MULTI-OMICS INTEGRATION
    Link with spatial features
    Integrate with tumor microenvironment
    Correlate with clinical outcomes

4. VISUALIZATION & EXPLORATION
    Joint embedding (UMAP/tSNE)
    Co-expression/co-flux networks
    Spatial maps of metabolic state

5. BIOLOGICAL VALIDATION
    Check if high-expression = high-flux
    Identify decoupling (post-transcriptional regulation)
    Pathway-specific patterns
"""

print(use_cases)

print("Part 6: Quick Example - Joint Visualisation")

from sklearn.manifold import TSNE

print("\nCreating t-SNE embedding of multi-modal space...")

# Subsample for speed
n_samples = min(2000, len(multimodal_scaled))
np.random.seed(42)
sample_idx = np.random.choice(len(multimodal_scaled), n_samples, replace=False)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_coords = tsne.fit_transform(multimodal_scaled[sample_idx])

# Load target for coloring
target = pd.read_csv('Patient_1_target_proliferation.csv', index_col=0)
target_matched = target.loc[common_spots]
y_sample = target_matched.iloc[sample_idx].values.ravel()

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Colored by proliferation
scatter = axes[0].scatter(tsne_coords[:, 0], tsne_coords[:, 1], 
                         c=y_sample, cmap='RdBu_r', s=10, alpha=0.6)
axes[0].set_title('Multi-Modal Metabolic Space\n(colored by proliferation)', 
                  fontweight='bold')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
plt.colorbar(scatter, ax=axes[0], label='Proliferation')

# Panel B: Expression-only t-SNE
expr_scaled = scaler.fit_transform(expr_matched)
expr_sample = expr_scaled[sample_idx]
tsne_expr = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_coords_expr = tsne_expr.fit_transform(expr_sample)

scatter2 = axes[1].scatter(tsne_coords_expr[:, 0], tsne_coords_expr[:, 1],
                          c=y_sample, cmap='RdBu_r', s=10, alpha=0.6)
axes[1].set_title('Expression-Only Space\n(for comparison)', fontweight='bold')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
plt.colorbar(scatter2, ax=axes[1], label='Proliferation')

# Panel C: Feature importance comparison
feature_vars = multimodal_data.var()
top_features = feature_vars.nlargest(15)

colors = ['blue' if 'Expr_' in f else 'red' for f in top_features.index]
axes[2].barh(range(len(top_features)), top_features.values, color=colors)
axes[2].set_yticks(range(len(top_features)))
axes[2].set_yticklabels([f.replace('Expr_', 'E:').replace('Flux_', 'F:') 
                         for f in top_features.index], fontsize=9)
axes[2].set_xlabel('Variance')
axes[2].set_title('Top 15 Variable Features', fontweight='bold')
axes[2].legend(['Expression', 'Flux'], loc='lower right')

plt.tight_layout()
plt.savefig('./Multimodal_Metabolic_Space.png', dpi=300, bbox_inches='tight')
print(" Saved: Multimodal_Metabolic_Space.png")
plt.show()


In [ ]:
"""
Run scFEA for Patient 2
Following the exact same workflow as Patient 1
"""

import os
import pandas as pd
from pathlib import Path
import shutil

print("RUNNING scFEA FOR PATIENT 2")


#  Check if scFEA exists
if not Path('./scFEA').exists():
    print("\nCloning scFEA repository...")
    os.system('git clone https://github.com/changwn/scFEA.git')
    print("scFEA cloned successfully")
else:
    print("\nscFEA already exists")

#  Create directories
Path('./scFEA/input').mkdir(parents=True, exist_ok=True)
Path('./scFEA/results/Patient_2').mkdir(parents=True, exist_ok=True)
print("Created required directories")

#  Copy expression file to scFEA input directory
source_expr = './data/Patient_2_expression.csv'
dest_expr = './scFEA/input/Patient_2_expression.csv'

if Path(source_expr).exists():
    shutil.copy(source_expr, dest_expr)
    print(f"\nCopied: {source_expr} -> {dest_expr}")
else:
    print(f"\nERROR: Expression file not found: {source_expr}")
    print("Please run the H5 conversion first!")

#  Run scFEA

print("RUNNING scFEA")

print("\nThis will take approximately 10-30 minutes")
print("You will see epoch updates as it trains\n")

cmd = """
python ./scFEA/src/scFEA.py \
    --data_dir ./scFEA/data \
    --input_dir ./scFEA/input \
    --res_dir ./scFEA/results/Patient_2 \
    --test_file Patient_2_expression.csv \
    --moduleGene_file module_gene_m168.csv \
    --stoichiometry_matrix cmMat_c70_m168.csv \
    --output_flux_file Patient_2_flux.csv \
    --output_balance_file Patient_2_balance.csv \
    --sc_imputation True
"""

exit_code = os.system(cmd)

if exit_code == 0:
    print("\n" + "="*70)
    print("scFEA COMPLETED SUCCESSFULLY")
    print("="*70)
else:
    print(f"\nscFEA failed with exit code: {exit_code}")
    print("Check error messages above")

#  Verify output files exist
print("\nVerifying output files")

flux_file = './scFEA/results/Patient_2/Patient_2_flux.csv'
balance_file = './scFEA/results/Patient_2/Patient_2_balance.csv'

if Path(flux_file).exists():
    print(f"SUCCESS: {flux_file}")
    flux_df = pd.read_csv(flux_file, index_col=0)
    print(f"  Shape: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")
    print(f"  First 5 modules: {list(flux_df.index[:5])}")
else:
    print(f"ERROR: Flux file not found")
    
    # Sometimes files are saved in current directory, try to move them
    if Path('./Patient_2_flux.csv').exists():
        print("  Found in current directory, moving...")
        shutil.move('./Patient_2_flux.csv', flux_file)
        print(f"  Moved to: {flux_file}")

if Path(balance_file).exists():
    print(f"SUCCESS: {balance_file}")
else:
    if Path('./Patient_2_balance.csv').exists():
        print("  Found in current directory, moving...")
        shutil.move('./Patient_2_balance.csv', balance_file)
        print(f"  Moved to: {balance_file}")

print("PATIENT 2 scFEA COMPLETE!")
print("\nNext step: Extract pathway-level flux features")

In [ ]:
"""
Run scFEA for Patient 3
Following the exact same workflow as Patient 1 & 2
"""

import os
import pandas as pd
from pathlib import Path
import shutil


print("RUNNING scFEA FOR PATIENT 3")


#  Create directories
Path('./scFEA/input').mkdir(parents=True, exist_ok=True)
Path('./scFEA/results/Patient_3').mkdir(parents=True, exist_ok=True)
print("Created required directories")

# Copy expression file to scFEA input directory
source_expr = './data/Patient_3_expression.csv'
dest_expr = './scFEA/input/Patient_3_expression.csv'

if Path(source_expr).exists():
    shutil.copy(source_expr, dest_expr)
    print(f"\nCopied: {source_expr} -> {dest_expr}")
else:
    print(f"\nERROR: Expression file not found: {source_expr}")

# Run scFEA
print("\n" + "="*70)
print("RUNNING scFEA")
print("="*70)
print("\nThis will take approximately 10-30 minutes")
print("You will see epoch updates as it trains\n")

cmd = """
python ./scFEA/src/scFEA.py \
    --data_dir ./scFEA/data \
    --input_dir ./scFEA/input \
    --res_dir ./scFEA/results/Patient_3 \
    --test_file Patient_3_expression.csv \
    --moduleGene_file module_gene_m168.csv \
    --stoichiometry_matrix cmMat_c70_m168.csv \
    --output_flux_file Patient_3_flux.csv \
    --output_balance_file Patient_3_balance.csv \
    --sc_imputation True
"""

exit_code = os.system(cmd)

if exit_code == 0:
    print("\n" + "="*70)
    print("scFEA COMPLETED SUCCESSFULLY")
    print("="*70)
else:
    print(f"\nscFEA failed with exit code: {exit_code}")

#  Verify output files exist
print("\nVerifying output files")

flux_file = './scFEA/results/Patient_3/Patient_3_flux.csv'
balance_file = './scFEA/results/Patient_3/Patient_3_balance.csv'

if Path(flux_file).exists():
    print(f"SUCCESS: {flux_file}")
    flux_df = pd.read_csv(flux_file, index_col=0)
    print(f"  Shape: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")
else:
    print(f"ERROR: Flux file not found")
    # Try to move from current directory
    if Path('./Patient_3_flux.csv').exists():
        print("  Found in current directory, moving...")
        shutil.move('./Patient_3_flux.csv', flux_file)
        print(f"  Moved to: {flux_file}")

if Path(balance_file).exists():
    print(f"SUCCESS: {balance_file}")
else:
    if Path('./Patient_3_balance.csv').exists():
        shutil.move('./Patient_3_balance.csv', balance_file)


In [ ]:
"""
Run scFEA for Patient 4
Following the exact same workflow as Patients 1, 2, 3
"""

import os
import pandas as pd
from pathlib import Path
import shutil

print("RUNNING scFEA FOR PATIENT 4")


#  Create directories
Path('./scFEA/input').mkdir(parents=True, exist_ok=True)
Path('./scFEA/results/Patient_4').mkdir(parents=True, exist_ok=True)
print("Created required directories")

#  Copy expression file to scFEA input directory
source_expr = './data/Patient_4_expression.csv'
dest_expr = './scFEA/input/Patient_4_expression.csv'

if Path(source_expr).exists():
    shutil.copy(source_expr, dest_expr)
    print(f"\nCopied: {source_expr} -> {dest_expr}")
else:
    print(f"\nERROR: Expression file not found: {source_expr}")

#  Run scFEA
print("RUNNING scFEA")
print("\nThis will take approximately 10-30 minutes")
print("You will see epoch updates as it trains\n")

cmd = """
python ./scFEA/src/scFEA.py \
    --data_dir ./scFEA/data \
    --input_dir ./scFEA/input \
    --res_dir ./scFEA/results/Patient_4 \
    --test_file Patient_4_expression.csv \
    --moduleGene_file module_gene_m168.csv \
    --stoichiometry_matrix cmMat_c70_m168.csv \
    --output_flux_file Patient_4_flux.csv \
    --output_balance_file Patient_4_balance.csv \
    --sc_imputation True
"""

exit_code = os.system(cmd)

if exit_code == 0:
    print("scFEA COMPLETED SUCCESSFULLY")
else:
    print(f"\nscFEA failed with exit code: {exit_code}")

# Verify output files exist
print("\nVerifying output files")

flux_file = './scFEA/results/Patient_4/Patient_4_flux.csv'
balance_file = './scFEA/results/Patient_4/Patient_4_balance.csv'

if Path(flux_file).exists():
    print(f"SUCCESS: {flux_file}")
    flux_df = pd.read_csv(flux_file, index_col=0)
    print(f"  Shape: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")
else:
    print(f"ERROR: Flux file not found")
    # Try to move from current directory
    if Path('./Patient_4_flux.csv').exists():
        print("  Found in current directory, moving...")
        shutil.move('./Patient_4_flux.csv', flux_file)
        print(f"  Moved to: {flux_file}")

if Path(balance_file).exists():
    print(f"SUCCESS: {balance_file}")
else:
    if Path('./Patient_4_balance.csv').exists():
        shutil.move('./Patient_4_balance.csv', balance_file)

print("PATIENT 4 scFEA COMPLETE!")

In [ ]:
"""
Run scFEA for Patient 5
Following the exact same workflow as Patients 1, 2, 3, 4
"""

import os
import pandas as pd
from pathlib import Path
import shutil

print("RUNNING scFEA FOR PATIENT 5")

# Create directories
Path('./scFEA/input').mkdir(parents=True, exist_ok=True)
Path('./scFEA/results/Patient_5').mkdir(parents=True, exist_ok=True)
print("Created required directories")

# Copy expression file to scFEA input directory
source_expr = './data/Patient_5_expression.csv'
dest_expr = './scFEA/input/Patient_5_expression.csv'

if Path(source_expr).exists():
    shutil.copy(source_expr, dest_expr)
    print(f"\nCopied: {source_expr} -> {dest_expr}")
else:
    print(f"\nERROR: Expression file not found: {source_expr}")

# Run scFEA
print("RUNNING scFEA")
print("\nThis will take approximately 10-30 minutes")
print("You will see epoch updates as it trains\n")

cmd = """
python ./scFEA/src/scFEA.py \
    --data_dir ./scFEA/data \
    --input_dir ./scFEA/input \
    --res_dir ./scFEA/results/Patient_5 \
    --test_file Patient_5_expression.csv \
    --moduleGene_file module_gene_m168.csv \
    --stoichiometry_matrix cmMat_c70_m168.csv \
    --output_flux_file Patient_5_flux.csv \
    --output_balance_file Patient_5_balance.csv \
    --sc_imputation True
"""

exit_code = os.system(cmd)

if exit_code == 0:
    print("scFEA COMPLETED SUCCESSFULLY")
else:
    print(f"\nscFEA failed with exit code: {exit_code}")

# Verify output files exist
print("\nVerifying output files")

flux_file = './scFEA/results/Patient_5/Patient_5_flux.csv'
balance_file = './scFEA/results/Patient_5/Patient_5_balance.csv'

if Path(flux_file).exists():
    print(f"SUCCESS: {flux_file}")
    flux_df = pd.read_csv(flux_file, index_col=0)
    print(f"  Shape: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")
else:
    print(f"ERROR: Flux file not found")
    # Try to move from current directory
    if Path('./Patient_5_flux.csv').exists():
        print("  Found in current directory, moving...")
        shutil.move('./Patient_5_flux.csv', flux_file)
        print(f"  Moved to: {flux_file}")

if Path(balance_file).exists():
    print(f"SUCCESS: {balance_file}")
else:
    if Path('./Patient_5_balance.csv').exists():
        shutil.move('./Patient_5_balance.csv', balance_file)

print("PATIENT 5 scFEA COMPLETE!")

In [ ]:
"""
Run scFEA for Patient 6
Following the exact same workflow as Patients 1, 2, 3, 4, 5
"""

import os
import pandas as pd
from pathlib import Path
import shutil

print("RUNNING scFEA FOR PATIENT 6")


# Create directories
Path('./scFEA/input').mkdir(parents=True, exist_ok=True)
Path('./scFEA/results/Patient_6').mkdir(parents=True, exist_ok=True)
print("Created required directories")

# Copy expression file to scFEA input directory
source_expr = './data/Patient_6_expression.csv'
dest_expr = './scFEA/input/Patient_6_expression.csv'

if Path(source_expr).exists():
    shutil.copy(source_expr, dest_expr)
    print(f"\nCopied: {source_expr} -> {dest_expr}")
else:
    print(f"\nERROR: Expression file not found: {source_expr}")

#  Run scFEA
print("RUNNING scFEA")
print("\nThis will take approximately 10-30 minutes")
print("You will see epoch updates as it trains\n")

cmd = """
python ./scFEA/src/scFEA.py \
    --data_dir ./scFEA/data \
    --input_dir ./scFEA/input \
    --res_dir ./scFEA/results/Patient_6 \
    --test_file Patient_6_expression.csv \
    --moduleGene_file module_gene_m168.csv \
    --stoichiometry_matrix cmMat_c70_m168.csv \
    --output_flux_file Patient_6_flux.csv \
    --output_balance_file Patient_6_balance.csv \
    --sc_imputation True
"""

exit_code = os.system(cmd)

if exit_code == 0:

    print("scFEA COMPLETED SUCCESSFULLY")
else:
    print(f"\nscFEA failed with exit code: {exit_code}")

#  Verify output files exist
print("\nVerifying output files")

flux_file = './scFEA/results/Patient_6/Patient_6_flux.csv'
balance_file = './scFEA/results/Patient_6/Patient_6_balance.csv'

if Path(flux_file).exists():
    print(f"SUCCESS: {flux_file}")
    flux_df = pd.read_csv(flux_file, index_col=0)
    print(f"  Shape: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")
else:
    print(f"ERROR: Flux file not found")
    # Try to move from current directory
    if Path('./Patient_6_flux.csv').exists():
        print("  Found in current directory, moving...")
        shutil.move('./Patient_6_flux.csv', flux_file)
        print(f"  Moved to: {flux_file}")

if Path(balance_file).exists():
    print(f"SUCCESS: {balance_file}")
else:
    if Path('./Patient_6_balance.csv').exists():
        shutil.move('./Patient_6_balance.csv', balance_file)

print("PATIENT 6 scFEA COMPLETE!")

In [4]:
# STEP 1: Load Visium HD data and create expression file for scFEA

import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
import os
import shutil

print("PROCESSING VISIUM HD DATA FOR scFEA (PATIENT 7)")

# Load Visium HD data
print("\nLoading Visium HD h5 file...")
h5_path = 'Patient_7/Visium_HD_16um_filtered_feature_bc_matrix.h5'

adata_p7 = sc.read_10x_h5(h5_path)
adata_p7.var_names_make_unique()

print(f"Loaded: {adata_p7.n_obs} spots x {adata_p7.n_vars} genes")

# Basic QC
print("\nApplying QC filters...")
sc.pp.filter_cells(adata_p7, min_counts=500)
sc.pp.filter_genes(adata_p7, min_cells=5)

print(f"After QC: {adata_p7.n_obs} spots x {adata_p7.n_vars} genes")

# Downsample if too many spots (scFEA is slow with >10k spots)
MAX_SPOTS = 10000
if adata_p7.n_obs > MAX_SPOTS:
    print(f"\nDownsampling from {adata_p7.n_obs} to {MAX_SPOTS} spots for scFEA...")
    np.random.seed(42)
    idx = np.random.choice(adata_p7.n_obs, MAX_SPOTS, replace=False)
    adata_p7 = adata_p7[idx, :].copy()
    print(f"After downsampling: {adata_p7.n_obs} spots")

# Normalize
print("Normalizing...")
sc.pp.normalize_total(adata_p7, target_sum=1e4)
sc.pp.log1p(adata_p7)

# For scFEA, we need normalized counts (not log-transformed)
print("\nPreparing expression matrix for scFEA...")

# Reload to get raw counts
adata_raw = sc.read_10x_h5(h5_path)
adata_raw.var_names_make_unique()

# Apply same QC filters
sc.pp.filter_cells(adata_raw, min_counts=500)
sc.pp.filter_genes(adata_raw, min_cells=5)

# Use same spots as downsampled adata_p7
adata_raw = adata_raw[adata_p7.obs_names, :].copy()

# Normalize but don't log-transform
sc.pp.normalize_total(adata_raw, target_sum=1e4)

print(f"Expression matrix: {adata_raw.n_obs} spots x {adata_raw.n_vars} genes")

# Convert to DataFrame with explicit list conversion to avoid pandas issues
if hasattr(adata_raw.X, 'toarray'):
    expr_matrix = adata_raw.X.toarray()
else:
    expr_matrix = np.array(adata_raw.X)

# Create DataFrame with explicit index/column conversion
gene_names = list(adata_raw.var_names)
spot_names = list(adata_raw.obs_names)

expr_df = pd.DataFrame(
    expr_matrix.T,
    index=gene_names,
    columns=spot_names
)

# Save expression file
Path('./data').mkdir(parents=True, exist_ok=True)
output_file = './data/Patient_7_expression.csv'

print(f"Saving expression file ({expr_df.shape[0]} genes x {expr_df.shape[1]} spots)...")
expr_df.to_csv(output_file)

print(f"Saved: {output_file}")

# Clean up
del adata_raw, expr_matrix, expr_df
import gc
gc.collect()

print("\nExpression file created successfully")

PROCESSING VISIUM HD DATA FOR scFEA (PATIENT 7)

Loading Visium HD h5 file...
Loaded: 167545 spots x 18085 genes

Applying QC filters...
After QC: 60723 spots x 16779 genes

Downsampling from 60723 to 10000 spots for scFEA...
After downsampling: 10000 spots
Normalizing...

Preparing expression matrix for scFEA...
Expression matrix: 10000 spots x 16779 genes
Saving expression file (16779 genes x 10000 spots)...
Saved: ./data/Patient_7_expression.csv

Expression file created successfully


In [ ]:
# PROCESS ALL PATIENTS' FLUX DATA (WITH VARIANCE CHECKING)

import pandas as pd
import numpy as np
from pathlib import Path

def process_flux_data(patient_id):
    """
    Process scFEA flux for one patient with variance checking.
    Returns raw modules if aggregation fails.
    """
    
    print(f"\n{'='*70}")
    print(f"Processing Patient {patient_id}")
    print(f"{'='*70}")
    
    # Load flux
    flux_file = f'./scFEA/results/Patient_{patient_id}/Patient_{patient_id}_flux.csv'
    
    if not Path(flux_file).exists():
        print(f"ERROR: Flux file not found: {flux_file}")
        return None
    
    flux_df = pd.read_csv(flux_file, index_col=0).T  # modules x spots
    print(f"Loaded flux: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")
    
    # Check variance
    module_variances = flux_df.var(axis=1)
    modules_with_variance = (module_variances > 1e-10).sum()
    
    print(f"Modules with variance: {modules_with_variance} / {len(module_variances)}")
    
    if modules_with_variance < 10:
        print("  WARNING: Almost no variance! scFEA aggregation will fail.")
        print("   Solution: Using RAW modules instead")
        
        # Filter to modules with variance
        flux_filtered = flux_df.loc[module_variances > 1e-10]
        flux_output = flux_filtered.T  # spots x modules
        output_file = f'./Patient_{patient_id}_flux_RAW_modules.csv'
        
        print(f" Saving {flux_output.shape[0]} spots x {flux_output.shape[1]} modules")
        
    else:
        print(" Good variance! Aggregating into pathways")
        
        # scFEA pathway mapping
        pathway_mapping = {
            'Glycolysis': ['M_1', 'M_2', 'M_3', 'M_4'],
            'Pyruvate_Metabolism': ['M_5', 'M_6'],
            'TCA_Cycle': ['M_7', 'M_8', 'M_9', 'M_10', 'M_11', 'M_12', 'M_13'],
            'Oxidative_Phosphorylation': ['M_14', 'M_15', 'M_16', 'M_17', 'M_18'],
            'Pentose_Phosphate': ['M_19', 'M_20', 'M_21'],
            'Fatty_Acid_Metabolism': ['M_22', 'M_23', 'M_24', 'M_25', 'M_26', 
                                      'M_27', 'M_28', 'M_29', 'M_30', 'M_31', 
                                      'M_32', 'M_33'],
            'Amino_Acid_Metabolism': ['M_34', 'M_35', 'M_36', 'M_37', 'M_38', 
                                      'M_39', 'M_40', 'M_41', 'M_42', 'M_43', 
                                      'M_44', 'M_45', 'M_46', 'M_47', 'M_48', 
                                      'M_49', 'M_50', 'M_51', 'M_52', 'M_53', 
                                      'M_54', 'M_55', 'M_56', 'M_57']
        }
        
        # Aggregate
        pathway_flux = {}
        for pathway_name, modules in pathway_mapping.items():
            available = [m for m in modules if m in flux_df.index]
            if available:
                # Mean across modules for each spot
                pathway_flux[f'Flux_{pathway_name}'] = flux_df.loc[available].mean(axis=0)
        
        flux_output = pd.DataFrame(pathway_flux)  # spots x pathways
        output_file = f'./Patient_{patient_id}_flux_metabolic_matched.csv'
        
        print(f" Saving {flux_output.shape[0]} spots x {flux_output.shape[1]} pathways")
    
    # Save
    flux_output.to_csv(output_file)
    print(f"Saved: {output_file}")
    
    # Show statistics
    print(f"\nFlux statistics:")
    print(f"  Mean: {flux_output.mean().mean():.6f}")
    print(f"  Std:  {flux_output.std().mean():.6f}")
    print(f"  Min:  {flux_output.min().min():.6f}")
    print(f"  Max:  {flux_output.max().max():.6f}")
    
    return flux_output


# PROCESS ALL PATIENTS
print("PROCESSING FLUX DATA FOR ALL PATIENTS")


results = {}
for patient_id in range(1, 7):
    try:
        flux_data = process_flux_data(patient_id)
        if flux_data is not None:
            results[f'Patient_{patient_id}'] = flux_data
    except Exception as e:
        print(f"\nERROR processing Patient {patient_id}: {e}")

print("SUMMARY")

for patient_id, flux_data in results.items():
    print(f"\n{patient_id}:")
    print(f"  Shape: {flux_data.shape}")
    print(f"  Features: {list(flux_data.columns[:5])}...")
    print(f"  Variance check: {'PASS' if flux_data.var().mean() > 1e-10 else 'FAIL'}")

In [ ]:
import pandas as pd

print("DIAGNOSING scFEA INPUT")

# Check the input file you gave to scFEA
input_file = './scFEA/input/Patient_1_expression.csv'  

expr_input = pd.read_csv(input_file, index_col=0)

print(f"\nInput file: {input_file}")
print(f"Shape: {expr_input.shape}")
print(f"\nFirst dimension (index): {expr_input.index.name or 'unnamed'}")
print(f"  First 5: {list(expr_input.index[:5])}")
print(f"\nSecond dimension (columns): {expr_input.columns.name or 'unnamed'}")
print(f"  First 5: {list(expr_input.columns[:5])}")

# Check if it's genes x cells or cells x genes
if len(expr_input.index) < len(expr_input.columns):
    print("\n  WARNING: More columns than rows!")
    print("   scFEA needs: GENES (rows) x CELLS (columns)")
    print("   You have: fewer rows than columns")
    print("\n   SOLUTION: Transpose your input!")

# Check gene names
print(f"\n\nChecking index (should be gene names):")
sample_indices = expr_input.index[:10]
for idx in sample_indices:
    print(f"  {idx}")

if any('-' in str(idx) for idx in sample_indices):
    print("\n  Index looks like barcodes (e.g., ACGT-1)")
    print("   These should be GENE NAMES (e.g., TP53, MYC)")
    print("\n   SOLUTION: Your data is transposed!")

# Check data values
print(f"\n\nData value range:")
print(f"  Min: {expr_input.min().min():.6f}")
print(f"  Max: {expr_input.max().max():.6f}")
print(f"  Mean: {expr_input.mean().mean():.6f}")

if expr_input.min().min() < 0:
    print("\n  Negative values detected!")
    print("   scFEA needs non-negative counts")
    print("   SOLUTION: Use raw or log1p-transformed counts")

if expr_input.max().max() > 100:
    print("\n Values look like counts (good)")
elif expr_input.max().max() < 10:
    print("\n Values look like log-normalized (acceptable)")
else:
    print("\n Values in unusual range")


In [ ]:
import pandas as pd

input_file = './scFEA/input/Patient_1_expression.csv'
expr_df = pd.read_csv(input_file, index_col=0)

print(f"Current scFEA input shape: {expr_df.shape}")
print(f"Genes: {expr_df.shape[0]:,}")
print(f"Spots: {expr_df.shape[1]:,}")

if expr_df.shape[0] < 5000:
    print("\n PROBLEM: Only", expr_df.shape[0], "genes!")
    print("   Need to fix Cell 9 and regenerate input")
else:
    print("\n GOOD:", expr_df.shape[0], "genes")